[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Continuous Integration &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the scratch folder, `run`, and the settings git needs, and the cell after it
writes the project's workflow as the notebook's worked examples left it. Run them first, then the
tasks in any order. The last cell removes the scratch folder.


In [1]:
import itertools
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import yaml

SCRATCH = Path("scratch")
PROJECT = SCRATCH / "stations"
(PROJECT / ".github" / "workflows").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun
os.environ["GIT_CONFIG_GLOBAL"] = os.devnull  # git reads neither this computer's settings for its user
os.environ["GIT_CONFIG_NOSYSTEM"] = "1"       # nor its settings for every user
for role in ["AUTHOR", "COMMITTER"]:          # and puts a practice name on the notebook's commits
    os.environ[f"GIT_{role}_NAME"] = "Weather Stations"
    os.environ[f"GIT_{role}_EMAIL"] = "stations@example.com"


def run(*command, folder=SCRATCH, environment=None):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True,
                              env=environment)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


print("ready:", SCRATCH)


ready: scratch


In [2]:
%%writefile scratch/stations/.github/workflows/tests.yml
name: Tests

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      matrix:
        os: [ubuntu-latest, macos-latest]
        python-version: ["3.12", "3.13", "3.14"]
    steps:
      - uses: actions/checkout@v7

      - uses: actions/setup-python@v7
        with:
          python-version: ${{ matrix.python-version }}

      - name: Install the project
        run: python -m pip install -e ".[test]"

      - name: Run the tests
        run: python -m pytest

      - name: Build a wheel
        run: python -m pip wheel --no-deps -w dist .


Writing scratch/stations/.github/workflows/tests.yml


**1.** The steps that run a command.


In [3]:
workflow = yaml.safe_load((PROJECT / ".github" / "workflows" / "tests.yml").read_text())

for step in workflow["jobs"]["test"]["steps"]:
    if "run" in step:
        print(f"{step['name']}: {step['run']}")


Install the project: python -m pip install -e ".[test]"
Run the tests: python -m pytest
Build a wheel: python -m pip wheel --no-deps -w dist .


The two steps without `run` use actions, and have no `name`, which is why the loop checks for `run`
before it reads `name`.


**2.** A script with and without `-e`.


In [4]:
for shell in [["bash", "-e", "-c"], ["bash", "-c"]]:
    code, printed = run(*shell, "false\necho done")
    print(" ".join(shell), "| exit code", code, "| printed:", repr(printed))


bash -e -c | exit code 1 | printed: ''
bash -c | exit code 0 | printed: 'done'


Without `-e`, bash carries on after `false`, and the script ends with the exit code of its last
command, `echo`, which is 0. A runner's shell uses `-e` so that a failing line fails the step.


**3.** A clone holds only what was committed.


In [5]:
repository, clone = SCRATCH / "task-repo", SCRATCH / "task-clone"
repository.mkdir()
(repository / "committed.txt").write_text("in the commit\n")
run("git", "init", "-q", "-b", "main", folder=repository)
run("git", "add", "committed.txt", folder=repository)
run("git", "commit", "-q", "-m", "One file", folder=repository)
(repository / "forgotten.txt").write_text("never added\n")

run("git", "clone", "-q", repository.resolve(), clone.name)
print("in the repository's folder:", sorted(path.name for path in repository.iterdir() if path.name != ".git"))
print("in the clone:              ", sorted(path.name for path in clone.iterdir() if path.name != ".git"))


in the repository's folder: ['committed.txt', 'forgotten.txt']
in the clone:               ['committed.txt']


`forgotten.txt` is in the folder and not in the clone, since a clone copies commits, and the file was
never in one.


**4.** An exit code hidden by `|| true`.


In [6]:
tests = SCRATCH / "task-tests"
tests.mkdir()
(tests / "test_sums.py").write_text("def test_one_and_one():\n    assert 1 + 1 == 3\n")

settings = {**os.environ, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
for suffix in ["", " || true"]:
    script = f'"{sys.executable}" -m pytest -q{suffix}'
    code, printed = run("bash", "-e", "-c", script, folder=tests, environment=settings)
    print(f"pytest -q{suffix:<9} exit code {code}")


pytest -q          exit code 1
pytest -q || true  exit code 0


The test fails both times, and only the plain command says so. The script names the notebook's own
Python by its path, since `python` on this computer's `PATH` may be another one.


**5.** A third system in the matrix.


In [7]:
matrix = dict(workflow["jobs"]["test"]["strategy"]["matrix"])
matrix["os"] = matrix["os"] + ["windows-latest"]

jobs = list(itertools.product(*matrix.values()))
print(len(jobs), "jobs")
for values in jobs:
    print(" ", dict(zip(matrix, values)))


9 jobs
  {'os': 'ubuntu-latest', 'python-version': '3.12'}
  {'os': 'ubuntu-latest', 'python-version': '3.13'}
  {'os': 'ubuntu-latest', 'python-version': '3.14'}
  {'os': 'macos-latest', 'python-version': '3.12'}
  {'os': 'macos-latest', 'python-version': '3.13'}
  {'os': 'macos-latest', 'python-version': '3.14'}
  {'os': 'windows-latest', 'python-version': '3.12'}
  {'os': 'windows-latest', 'python-version': '3.13'}
  {'os': 'windows-latest', 'python-version': '3.14'}


Three systems and three Pythons make nine jobs. `dict(...)` copies the matrix, so the workflow read
from the file keeps its two systems.


**6.** A badge for one branch.


In [8]:
page = "https://github.com/johnfisher-ai/Python-Visual-Guides/actions/workflows/notebooks.yml"

print(f"[![notebooks.yml]({page}/badge.svg?branch=main)]({page})")


[![notebooks.yml](https://github.com/johnfisher-ai/Python-Visual-Guides/actions/workflows/notebooks.yml/badge.svg?branch=main)](https://github.com/johnfisher-ai/Python-Visual-Guides/actions/workflows/notebooks.yml)


`?branch=main` goes after `badge.svg`, in the address of the image, and the link around it still
opens the workflow's page.

Last, remove the scratch folder:


In [9]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Continuous Integration](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/10-continuous-integration.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
